<a href="https://colab.research.google.com/github/Saleh-Furqan/OCR_playground/blob/benchmark-testing/assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Submission Instructions:**

Prerequisites: If you did not attend the in-person tutorial session, please ensure you review the materials for Tutorial 2 independently before starting this assignment.

Implementation: For each problem, implement your solution directly in this Jupyter Notebook.

Documentation: Provide clear explanations for your approach and include necessary comments within your code.

Execution: Run your code cells and **keep the output visible** (do not clear the results). We need to see the running results to grade your work. Please ensure that your code is fully runnable. Submissions with severe runtime errors or results that do not match the code execution will be flagged as potential plagiarism.

Submission: Pack the completed notebook file (.ipynb) and the result files (.gif, .json) with proper name into one zip file, and upload to the CUHK Blackboard platform.

**Important Notice:**  
This is an **individual** assignment. Any instances of identical code or writing may be flagged as potential plagiarism and will be subject to further investigation.

##(25pts) Problem 1: Visualizing 2D Convolution

Convolution is the fundamental operation behind Convolutional Neural Networks (CNNs). Unlike standard matrix multiplication, convolution acts as a **spatial filter**. A small matrix (the **kernel** or filter) slides over the input image, performing an element-wise multiplication and sum at each position to generate a feature map.

In this problem, you will implement the core logic of a 2D convolution from scratch and visualize the sliding window process.

### The Sliding Window Mechanism

Imagine a flashlight (the kernel) shining over a picture (the input). You slide the flashlight across the image:
1.  **Padding ($P$):** Sometimes we add a border of zeros around the input to control the output size.
2.  **Stride ($S$):** The step size of the flashlight. A stride of 1 means moving pixel-by-pixel; a stride of 2 means skipping every other pixel.
3.  **Kernel ($K$):** The filter that detects specific features (e.g., edges).

### Your Task

Complete the `animate_convolution` function below.

**Note:** The visualization code (matplotlib animation) is provided. Your job is to make the math work so the animation renders correctly.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
import numpy as np
from IPython.display import Image

# Set plotting style
plt.rcParams['image.cmap'] = 'gray'

# --- Provided Helper Functions and Setup ---

def plot_image(tensor, title=""):
    """Helper function to plot a single tensor image."""
    if tensor.dim() == 4:
        tensor = tensor.squeeze(0)  # Remove batch dim
    if tensor.dim() == 3 and tensor.shape[0] == 1:
        tensor = tensor.squeeze(0) # Remove channel dim if it's 1

    tensor = tensor.detach().numpy()
    plt.figure(figsize=(4, 4))
    plt.imshow(tensor, vmin=0, vmax=255) # Fix scale for consistency
    plt.title(title)
    plt.axis('off')
    plt.show()

# 1. Create the 32x32 sample image
input_image_tensor = torch.zeros(1, 1, 32, 32)  # Batch size 1, 1 Channel, H=32, W=32

# Add horizontal and vertical edges (using a slightly lower intensity so adding kernels don't explode overly)
input_image_tensor[:, :, 10:20, :] = 200
input_image_tensor[:, :, :, 10:20] = 200

print("Input Image Shape:", input_image_tensor.shape)
plot_image(input_image_tensor, "Original Input Image")


# --- Problem 1: Convolution Visualization ---

def animate_convolution(input_tensor, kernel_tensor, stride=1, padding=0, filename="conv_process.gif"):
    """
    Animates the 2D convolution process and saves it as a GIF.

    Args:
        input_tensor (torch.Tensor): Shape (1, 1, H_in, W_in).
        kernel_tensor (torch.Tensor): Shape (1, 1, K_H, K_W).
        stride (int): Stride of the convolution.
        padding (int): Zero-padding added to both sides of the input.
        filename (str): Output filename for the GIF.
    """

    # Ensure inputs are 4D tensors for consistency
    if input_tensor.dim() != 4 or kernel_tensor.dim() != 4:
        raise ValueError("Input and Kernel must be 4D tensors (B, C, H, W)")

    # Get dimensions
    batch, in_channels, h_in, w_in = input_tensor.shape
    k_out, k_in, k_h, k_w = kernel_tensor.shape

    assert batch == 1 and in_channels == 1 and k_out == 1 and k_in == 1, \
        "This visualization currently only supports single batch, single channel input and output."

    # 1. Apply Padding to the input image
    # Pytorch is allowed.
    ### TODO: YOUR CODE HERE (Start)
    padded_input =
    padded_h, padded_w =
    ### TODO: YOUR CODE HERE (End)

    # Convert tensors to numpy for easier plotting
    padded_input_np = padded_input.squeeze().detach().numpy()
    kernel_np = kernel_tensor.squeeze().detach().numpy()

    # ==============================================================================
    # Part A: Calculate Output Dimensions and Initialize Output
    # ==============================================================================
    # Students need to calculate these dimensions correctly.

    ### TODO: YOUR CODE HERE (Start: Calculate h_out and w_out) ###
    h_out =
    w_out =
    ### TODO: YOUR CODE HERE (End) ###

    print(f"Convolution Output Shape: ({h_out}, {w_out})")

    # Initialize the output image (feature map) with zeros, to be filled step-by-step
    # We initialize with a middle gray value just for visualization contrast initially
    output_image_np = np.zeros((h_out, w_out))

    # Total number of steps in the animation
    total_steps = h_out * w_out

    # ==============================================================================
    # Part B: Setting up the Matplotlib Figure for Animation
    # ==============================================================================
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

    # Left Subplot: Padded Input Image with Sliding Window
    ax1.imshow(padded_input_np, vmin=0, vmax=255, extent=[0, padded_w, padded_h, 0])
    ax1.set_title(f"Input (Padded: p={padding}) \n& Sliding Window (k={k_h}x{k_w}, s={stride})")
    # Create a red rectangle patch representing the kernel window. Initial position (0,0).
    rect = patches.Rectangle((0, 0), k_w, k_h, linewidth=2, edgecolor='r', facecolor='none')
    ax1.add_patch(rect)
    # Draw grid lines to show pixels clearly
    ax1.set_xticks(np.arange(0, padded_w, 1)); ax1.set_yticks(np.arange(0, padded_h, 1))
    ax1.grid(color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
    ax1.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False) # Hide ticks

    # Right Subplot: The accumulating Output Feature Map
    # We initialize plots. vmin/vmax define the color scale ranges.
    # We guess a range for visualization. Convolution outputs can go outside 0-255.
    # A safe bet for visualization is to check the possible range based on kernel weights.
    max_possible_val = np.sum(np.abs(kernel_np)) * 255
    vmin_out, vmax_out = -max_possible_val/2, max_possible_val/2 # Center around 0 for edge detectors

    im2 = ax2.imshow(output_image_np, vmin=vmin_out, vmax=vmax_out, extent=[0, w_out, h_out, 0])
    ax2.set_title(f"Output Feature Map\nResult accumulating...")
    ax2.set_xticks(np.arange(0, w_out, 1)); ax2.set_yticks(np.arange(0, h_out, 1))
    ax2.grid(color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
    ax2.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    plt.tight_layout()

    # ==============================================================================
    # Part C: The Animation Update Function
    # ==============================================================================
    def update(frame_idx):
        # 1. Convert linear frame index to 2D output coordinates (row_out, col_out)
        row_out = frame_idx // w_out
        col_out = frame_idx % w_out

        # 2. Calculate the top-left corner of the window on the PADDED input image
        # Students need to figure out how stride affects this index.
        ### TODO: YOUR CODE HERE (Start) ###
        h_start =
        w_start =


        h_end =
        w_end =
        ### TODO: YOUR CODE HERE (End) ###
        # --- Update Left Subplot (Visuals) ---
        # Move the red rectangle patch. Matplotlib uses (x, y) -> (col, row)
        rect.set_xy((w_start, h_start))

        # --- Update Right Subplot (Calculation) ---
        # Perform the convolution operation for this specific window region.
        # 1. Extract the region of interest (ROI) from the padded input.
        # 2. Perform element-wise multiplication with the kernel.
        # 3. Sum the results to get a single scalar value.

        ### TODO: YOUR CODE HERE (Start: Implement the convolution step) ###



        convolution_result =
        ### TODO: YOUR CODE HERE (End) ###

        # Update the corresponding pixel in the output image numpy array
        output_image_np[row_out, col_out] = convolution_result

        # Refresh the image data in the plot
        im2.set_data(output_image_np)

        return rect, im2

    # Create animation structure
    anim = FuncAnimation(fig, update, frames=total_steps, interval=100, blit=True)

    # Save the animation as a GIF using Pillow writer (no external ffmpeg needed)
    print(f"Generating GIF '{filename}' with {total_steps} frames. Please wait...")
    anim.save(filename, writer='pillow', fps=10)
    print("Done!")
    plt.close(fig) # Prevents the static final frame from showing in the notebook output
    return filename

# ==============================================================================
# Test Case: Define a Kernel and Run Animation
# ==============================================================================

# Let's define a simple vertical edge detection kernel (Sobel-like)
kernel_data = torch.tensor([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=torch.float32)

# Reshape to (Batch=1, Channel=1, H=3, W=3)
kernel_tensor = kernel_data.unsqueeze(0).unsqueeze(0)

print("Kernel Shape:", kernel_tensor.shape)
# print("Kernel Values:\n", kernel_tensor.squeeze())

# Define parameters
stride_param = 2
padding_param = 1
gif_filename = 'conv_demo_edge_s2_p1.gif'

# Run the animation function
# NOTE: This might take roughly 10-20 seconds to render depending on your machine.
gif_path = animate_convolution(
    input_image_tensor,
    kernel_tensor,
    stride=stride_param,
    padding=padding_param,
    filename=gif_filename
)

# Try different padding and stride, see what happens.
# Please include the gif file in your submission.


##(25pts) Problem 2: 1D Convolution for ECG Anomaly Detection

In the previous problem, you visualized 2D convolution on images. Now, let's look at **1D convolution**, which is widely used in signal processing (e.g., Audio, ECG, Stock prices).

### The Scenario
You are given a simplified ECG signal dataset.
1.  **Normal Heartbeat:** Represented by a smooth, low-frequency sine wave.
2.  **Anomaly (Arrhythmia):** Represented by a sudden, sharp spike (high-frequency noise) occurring at a random position in the signal.

### Your Task
Your goal is to **manually design a 1D Kernel** and a **Threshold** to detect these anomalies. You are **NOT** allowed to train a machine learning model. You must use the properties of convolution.

**Hint:** * A "smooth" signal changes slowly.
* A "spike" changes very abruptly.
* What kind of kernel values would output a **large response** only when there is a sudden change in the signal, and a **small response** when the signal is smooth? (Think about derivatives or edge detection).

### Dataset Format
We have generated 100 samples in `ecg_data.json`. Each entry contains:
* `id`: Sample ID
* `signal`: A list of 100 float values representing the ECG voltage.
* `label`: `0` for Normal, `1` for Abnormal.
* `anomaly_loc`: The index where the spike occurred (or `null` if normal).

In [ ]:
import numpy as np
import json
import torch
import torch.nn.functional as F

# Load the dataset
with open('ecg_data.json', 'r') as f:
    test_data = json.load(f)

def evaluate_detection(my_param):
    """
    Evaluates the user's kernel and threshold on the test_data.
    """
    correct_predictions = 0
    total_samples = len(test_data)


    for sample in test_data:
        signal = sample['signal']
        ground_truth_label = sample['label']


        #TODO: your algorithm here

        prediction = #0 for Normal, 1 for Abnormal.

        if prediction == ground_truth_label:
            correct_predictions += 1

    accuracy = correct_predictions / total_samples
    print(f"Final Accuracy: {accuracy * 100:.2f}%")

    if accuracy == 1.0:
        print("🎉 Excellent! Your kernel successfully detected all anomalies.")
    else:
        print("⚠️ Not quite there. Try adjusting your kernel values or threshold.")

# ==========================================
# TODO: DESIGN YOUR KERNEL, CONVOLUTION, MAYBE USE SOME THRESHOLD
# ==========================================

my_param =

# Run Evaluation
evaluate_detection(my_param)

## (25pts) Problem 3:  Image Compression

In standard image compression (like JPEG), we compress blocks of pixels. However, if we know the image consists of specific objects (in this case, digits 0, 7, and 8), we can achieve **extreme compression** by only storing the **metadata**:



### Your Task
1.  We have provided the binary shapes (templates) for 0, 7, and 8.
Assume the images are constructed perfectly from the templates without noise.
2.  You need to implement a compression function `compress_image(image, templates)` that compresses the sample file as small as you can.
3. Use 2D convolution!

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Load the dataset
# Ensure 'digit_compression_data.pth' is in the same directory
try:
    data = torch.load('digit_compression_data.pth')
    test_images = data['images']      # Shape: (100, 3, 32, 32)
    templates = data['templates']     # Shape: (3, 8, 8) - The digit template
    print("Data loaded successfully.")
except FileNotFoundError:
    print("Error: 'digit_compression_data.pth' not found.")

# --- Visualization Helper ---
def visualize_sample(index):
    image = test_images[index]

    # Convert to (H, W, C) for plotting
    img_np = image.permute(1, 2, 0).numpy()

    plt.figure(figsize=(4,4))
    plt.imshow(img_np)
    plt.title(f"Sample {index}")
    plt.axis('off')
    plt.show()

# Visualize the provided Templates (Kernels)
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for i in range(3):
    ax[i].imshow(templates[i], cmap='gray')
    ax[i].set_title(f"Template {i}")
    ax[i].axis('off')
plt.suptitle("The shapes of the 3 digits appeared in the data")
plt.show()

# Visualize one sample from the dataset
visualize_sample(0)

In [ ]:
#TODO: You code of the compression algorithm here.

#1. Compress 'digit_compression_data.pth' and save the compressed file

#2. Load your compressed file and decompress

#3. See if your compression algorithm achieve a lossless compression

##(20pts) Problem 4: Action Detection in Video (3D Convolution)

In the final problem, we move from static images to **Video**.
You are given video clips of a "Tank Game". The tank is a $3 \times 3$ square that moves **Up, Down, Left, or Right (there is no Still)** by 1 pixel at each step.

**Your Task:**
Recover the player's actions (e.g., `['UP', 'RIGHT', 'UP', ...]`) from the video using **3D Convolution**.

### Understanding 3D Convolution
In 2D convolution, the kernel slides over Height and Width.
In **3D convolution**, the kernel acts on a volume: **(Time, Height, Width)**.
Input Tensor Shape: $(N, C, D, H, W)$
* $N$: Batch Size
* $C$: Channels (1 for grayscale)
* $D$: Depth (Time / Number of Frames)
* $H, W$: Height, Width

### The "Motion Detector" Kernel
To detect motion, your kernel needs to span **2 frames** (Time depth = 2).
* **Frame $t$ (Current):** Should match the tank's *current* position.
* **Frame $t-1$ (Previous):** Should match the tank's *previous* position.


In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Load Data
try:
    data = torch.load('tank_video_data.pth')
    videos = data['video']  # Shape: (100, 1, 10, 16, 16) -> (N, C, video_sequence_length, H, W)
    labels = data['labels'] # List of lists of strings
    print(f"Loaded {len(videos)} video sequences.")
except FileNotFoundError:
    print("Error: 'tank_video_data.pth' not found.")

# Visualize 2 consecutive frames to see motion
def show_motion(video_tensor, frame_idx):
    # video_tensor: (1, Depth, H, W)
    f1 = video_tensor[0, frame_idx, :, :]
    f2 = video_tensor[0, frame_idx+1, :, :]

    fig, ax = plt.subplots(1, 2, figsize=(6, 3))
    ax[0].imshow(f1, cmap='gray', vmin=0, vmax=1)
    ax[0].set_title(f"Frame {frame_idx}")
    ax[0].grid(True, color='gray', alpha=0.3)

    ax[1].imshow(f2, cmap='gray', vmin=0, vmax=1)
    ax[1].set_title(f"Frame {frame_idx+1}")
    ax[1].grid(True, color='gray', alpha=0.3)
    plt.show()

print("Visualizing motion between Frame 0 and Frame 1 for Sample 0:")
show_motion(videos[0], 0)
print(f"Ground Truth Action: {labels[0][0]}")

#because we consider the relative moving direction, there are only video_sequence_length-1 labels for the motion.
print(f"length of label: {len(labels[0])}")

In [ ]:
# --- Problem 4: Implement 3D Convolution ---

def detect_actions(video):
    #TODO: Your algorithm here

    '''Hint: You might want to use 4 different kernels (one for each direction).
    For each time step, find which kernel gives the maximum activation across the entire spatial frame.'''








    return predicted_actions

# --- Evaluation Loop ---
print("Evaluating Action Detection...")
total_correct = 0
total_actions = 0

for i in range(len(videos)):
    vid = videos[i]       # (1, 10, 16, 16)
    true_acts = labels[i] # Length 9

    pred_acts = detect_actions(vid) # Should return length 9


    # Check if length matches
    if len(pred_acts) != len(true_acts):
        print(f"Error: Expected {len(true_acts)} actions, got {len(pred_acts)}")
        break

    for gt, pred in zip(true_acts, pred_acts):
        if gt == pred:
            total_correct += 1
        total_actions += 1



acc = total_correct / total_actions * 100
print(f"Total Accuracy: {acc:.2f}%")

if acc > 95:
    print("✅ Awesome! You built a working 3D motion detector.")
else:
    print("⚠️ Accuracy is low. Check your kernel coordinates logic.")

###(5pts) Problem 4.1:

If the speed of the tank movement is changed, will the current convolution algorithm work? Why or why not?